# Phase 2 — Engineer & Transform Features
**Goal:** Turn the clean data into a richer, model-ready dataset by creating new columns, encoding categories, scaling numerics, and removing redundant features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('data/cleaned/Ames_Housing_Cleaned.csv')
print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head(3)

## Step 1 — One-Hot Encoding (2 categorical columns)
**Why:** Linear models and correlation analysis require numeric inputs. One-hot encoding converts nominal categories into separate 0/1 columns without implying any ordinal ranking.  
- `MS Zoning` — the zoning classification (residential, commercial, etc.) is a strong price driver but has no natural order.
- `Sale Condition` — how the sale occurred (Normal, Abnormal, Partial, etc.) affects price but categories are not ordered.

In [ ]:
df = pd.get_dummies(df, columns=['MS Zoning', 'Sale Condition'], drop_first=True)
print(f'Columns after one-hot encoding: {df.shape[1]}')

## Step 2 — Ordinal Encoding (1 ordered column)
**Why:** `Exter Qual` has a natural order (Poor → Excellent). Assigning integers 1–5 preserves that ordering — unlike one-hot encoding which would discard it.

In [ ]:
quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1}
df['Exter Qual Num'] = df['Exter Qual'].map(quality_map)
print('Exter Qual ordinal mapping:')
print(df[['Exter Qual', 'Exter Qual Num']].drop_duplicates().sort_values('Exter Qual Num'))

## Step 3 — Domain Feature 1: Price per Square Foot
**Why:** Raw `SalePrice` doesn't tell you if a house is expensive relative to its size. `Price_per_SqFt` standardises value by area, making houses comparable regardless of total square footage. Safe division prevents dividing by zero.

In [ ]:
df['Price_per_SqFt'] = df['SalePrice'] / df['Gr Liv Area'].replace(0, np.nan)
print(f'Price_per_SqFt — mean: ${df["Price_per_SqFt"].mean():,.2f}, max: ${df["Price_per_SqFt"].max():,.2f}')

## Step 4 — Domain Feature 2: Total Bath Count
**Why:** Buyers care about total bathrooms, not just full or half separately. Combining them into one number (`Total_Bathrooms`) captures overall bathroom supply. A half-bath is worth roughly 0.5 of a full bath.

In [ ]:
df['Total_Bathrooms'] = (df['Full Bath'] + df['Bsmt Full Bath'] +
                         0.5 * df['Half Bath'] + 0.5 * df['Bsmt Half Bath'])
print(f'Total_Bathrooms — distribution:')
print(df['Total_Bathrooms'].value_counts().sort_index())

## Step 5 — Interaction Feature: Quality × Living Area
**Why:** A large house with poor quality is worth less than a smaller house with excellent quality. Multiplying `Overall Qual` × `Gr Liv Area` captures the combined luxury-and-space effect in a single feature.

In [ ]:
df['Qual_x_Area'] = df['Overall Qual'] * df['Gr Liv Area']
print(f'Qual_x_Area — mean: {df["Qual_x_Area"].mean():,.1f}, correlation with SalePrice: {df["Qual_x_Area"].corr(df["SalePrice"]):.3f}')

## Step 6 — Log Transform on Skewed Column
**Why:** `SalePrice` is right-skewed (a few very expensive houses pull the tail). `np.log1p()` compresses the tail and makes the distribution more symmetric — a requirement for linear regression to perform well.  
The before/after histograms below confirm the improvement.

In [ ]:
df['SalePrice_Log'] = np.log1p(df['SalePrice'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['SalePrice'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('SalePrice — Before Log Transform (Right-Skewed)')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['SalePrice_Log'], bins=50, color='seagreen', edgecolor='white')
axes[1].set_title('SalePrice_Log — After Log Transform (More Symmetric)')
axes[1].set_xlabel('log(Price + 1)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print(f'Skewness before: {df["SalePrice"].skew():.3f}')
print(f'Skewness after:  {df["SalePrice_Log"].skew():.3f}')

## Step 7 — Binning: House Age Category
**Why:** Raw `Year Built` is a continuous number, but analysts often think in eras. Binning into Old / Recent / New creates an interpretable categorical feature that may capture non-linear age effects.

In [ ]:
df['House_Age_Category'] = pd.cut(
    df['Year Built'],
    bins=[0, 1950, 2000, 2011],
    labels=['Old', 'Recent', 'New']
)
print('House age category distribution:')
print(df['House_Age_Category'].value_counts())

## Step 8 — StandardScaler on 2 Numerical Columns
**Why:** `Gr Liv Area` (in square feet, ~300–4000) and `Lot Area` (in square feet, up to 200,000) are on very different scales. StandardScaler transforms each to mean=0, std=1, so neither dominates distance-based calculations.

In [ ]:
scaler = StandardScaler()
df[['Gr Liv Area Scaled', 'Lot Area Scaled']] = scaler.fit_transform(
    df[['Gr Liv Area', 'Lot Area']]
)
print('Scaled columns — mean and std (should be ~0 and ~1):')
print(df[['Gr Liv Area Scaled', 'Lot Area Scaled']].describe().loc[['mean', 'std']].round(4))

## Step 9 — Remove Redundant Features (r > 0.95)
**Why:** Highly correlated columns carry the same information twice. They inflate the feature space and can cause multicollinearity in regression models. We keep only one of each near-duplicate pair.

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]

print(f'Columns to drop due to r > 0.95 correlation: {to_drop}')
df.drop(columns=to_drop, inplace=True)
print(f'Final column count after redundancy removal: {df.shape[1]}')

In [ ]:
import os
os.makedirs('data/cleaned', exist_ok=True)
df.to_csv('data/cleaned/Ames_Housing_Features.csv', index=False)
print('Feature-engineered data saved to data/cleaned/Ames_Housing_Features.csv')